In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.9.0 torchvision==0.15.1 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install transformers==4.30.2
!pip install sentencepiece==0.1.99
!pip install kobert-transformers

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
import json

# JSON 파일 열기
with open('/content/감성대화말뭉치(최종데이터)_Training.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 리스트 구조일 경우
dialogues = data

samples = []

for d in dialogues:
    try:
        emotion = d['profile']['emotion']['type']
        utterances = d['talk']['content']
        for key in utterances:
            if key.startswith('HS'):  # 사용자 발화만
                sentence = utterances[key].strip()
                # 빈 문장 제외
                if sentence != '':
                    samples.append({
                        'sentence': sentence,
                        'emotion': emotion
                    })
    except Exception as e:
        continue

# DataFrame으로 변환
df_train = pd.DataFrame(samples)



In [ ]:
# 접두사 기준 감정 매핑
prefix_map = {
    'E1': '분노',
    'E2': '슬픔',
    'E3': '불안',
    'E4': '상처',
    'E5': '당황',
    'E6': '기쁨'
}

# 감정명 → 긍/부정 매핑
label_map_a = {
    '기쁨': 'positive',
    '분노': 'negative',
    '슬픔': 'negative',
    '불안': 'negative',
    '당황': 'negative',
    '상처': 'negative'
}

# 감정 코드 → 감정명 매핑 함수
def map_emotion(code):
    return prefix_map.get(code[:2], None)

# 매핑 적용
df_train['emotion_name'] = df_train['emotion'].apply(map_emotion)
df_train['label'] = df_train['emotion_name'].map(label_map_a)


# 결측값 제거
df_a = df_train.dropna(subset=['label'])


In [ ]:
# 감정별 1000개씩 샘플링
df_sample = (
    df_a.groupby('label', group_keys=False)
        .apply(lambda x: x.sample(n=10000, random_state=42))
        .reset_index(drop=True)
)

# 결과 확인
print(df_sample['label'].value_counts())


label
negative    5000
positive    5000
Name: count, dtype: int64


/tmp/ipython-input-2177850658.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=5000, random_state=42))


In [ ]:
texts = df_sample['sentence'].tolist()
labels = df_sample['label'].tolist() # 긍정/부정 라벨

# label encoding
le = LabelEncoder()
labels = le.fit_transform(labels)

# train/val split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)


In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item

Epoch 1 / 3, Train Loss: 0.4416
Validation Accuracy: 0.8410
Epoch 2 / 3, Train Loss: 0.3520
Validation Accuracy: 0.8355
Epoch 3 / 3, Train Loss: 0.2499
Validation Accuracy: 0.8255
positive
negative


In [ ]:
# 토크나이저 + KoBERT 모델 로드
from kobert_transformers import get_tokenizer

tokenizer = get_tokenizer()  # KoBERT용 토크나이저 자동 로드

base_model = BertModel.from_pretrained('monologg/kobert')

# 분류용 헤드 추가
class KoBERTClassifier(nn.Module):
    def __init__(self, base_model, num_labels=2):
        super().__init__()
        self.bert = base_model
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        pooled_output = outputs.last_hidden_state[:,0,:]  # [CLS] 토큰
        logits = self.classifier(pooled_output)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
        return loss, logits

model = KoBERTClassifier(base_model)

# DataLoader
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)


# Optimizer & device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)


In [ ]:
#학습
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        loss, logits = model(input_ids, attention_mask, labels=labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} / {epochs}, Train Loss: {avg_loss:.4f}")

    # 검증
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            _, logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / total
    print(f"Validation Accuracy: {acc:.4f}")




In [ ]:
def predict_sentiment(text):
    model.eval()
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        _, logits = model(input_ids, attention_mask)
        pred = torch.argmax(logits, dim=1).item()
        return le.inverse_transform([pred])[0]


print(predict_sentiment("오늘 너무 기분 좋아!"))  # positive
print(predict_sentiment("짜증나고 우울하다."))      # negative

In [ ]:
# 모델 파라미터만 저장
torch.save(model.state_dict(), 'kobert_model_ver6.pt')


In [ ]:
#사용시
review = pd.read_csv('/content/reviews_merged_all.csv')
review['pred_label'] = review['content'].apply(predict_sentiment)

In [ ]:
review[['pred_label','app']].value_counts().sort_index()

pred_label  app          
negative    banapresso        447
            composecoffee    1484
            ediya            2571
            gongcha           973
            hollys           1188
            mammothcoffee     474
            megacoffee       2756
            paikdabang       1256
            starbucks        7526
            theventi          818
            tomntoms          786
            twosomeplace     2332
positive    banapresso        318
            composecoffee     471
            ediya            1821
            gongcha           184
            hollys            295
            mammothcoffee     148
            megacoffee       1214
            paikdabang        658
            starbucks        4117
            theventi          301
            tomntoms         1001
            twosomeplace      599
Name: count, dtype: int64

In [ ]:
review[review['pred_label']=='positive'].rating.mean()

,count
rating,
5,6262
1,2346
4,1170
3,620
2,341


In [ ]:
review[review['pred_label']=='negative'].rating.value_counts()

,count
rating,
1,17768
2,1941
3,1694
5,861
4,735


In [ ]:
review[review['pred_label']=='positive'][['content','rating']].head(30)

,content,rating
4,상품이 다양해서 좋았읍니다,5
7,무한로딩,1
8,깜빡하고 픽업안했는데 전혀 알림이 안왔네요 알림 허용됨 상태이고 배터리는 최적화여서...,1
9,"월 150,000 포인트 적립으로 주문 가능합니까?",5
12,부천,5
13,옵션에 연하게는 있는데 덜달게가 없어서 불편해요 ㅠㅠ 추가해주세요!,5
17,Pass인증 존나게 좋아하시네요 비번찾고도 폰 한개에서 앱 한개열어서 로그인 10분...,1
29,시럽들어가는 메뉴(꿀.헤이즐넛.바닐라)시럽절반만 넣을수 있게 선택사항좀 만들어주세요...,1
34,디자인이 깔끔한 쓰레기 어플.,1
36,업데이트 되더니 회원정보 삭제.. 모아논 스탬프 돌려주세요!!!!!!,1
